# LAMBADA activation cache
Run on a Colab GPU after saving the training manifest with dataset.ipynb. This pass keeps 5 examples from train, validation, and test. Each split gets a paired cache of final-position Llama and Qwen activations before and after RMSNorm.


In [16]:
%pip install -q "transformers>=5" "datasets>=3,<5" "huggingface_hub>=1.32,<2" accelerate safetensors tqdm

In [17]:
import gc, hashlib, json, os, re
from pathlib import Path
import torch
from datasets import load_dataset
from transformers import AutoModel, AutoTokenizer

In [18]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/coms-4705-project")
if not DRIVE_PROJECT_DIR.is_dir():
    raise RuntimeError("Add a My Drive shortcut to the shared coms-4705-project folder.")
print(DRIVE_PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/coms-4705-project


## Select the saved training set
Leave TRAIN_DIR unset if there is one saved manifest. With several runs, paste the directory printed by the dataset notebook.

In [19]:
TRAIN_DIR = None
BATCH_SIZE = 8
if TRAIN_DIR is None:
    manifests = sorted((DRIVE_PROJECT_DIR / "datasets/lambada/train").glob("*/examples.jsonl"))
    if len(manifests) != 1:
        raise ValueError(f"Set TRAIN_DIR to the desired saved training directory. Found: {manifests}")
    TRAIN_DIR = manifests[0].parent
train_dir = Path(TRAIN_DIR)
meta = json.loads((train_dir / "provenance.json").read_text())
manifest = (train_dir / "examples.jsonl").read_bytes()
assert hashlib.sha256(manifest).hexdigest() == meta["manifest_sha256"]
rows_by_split = {"train": [json.loads(line) for line in manifest.splitlines()]}
assert len(rows_by_split["train"]) == meta["n_train"]
N_PER_SPLIT = 5
rows_by_split["train"] = rows_by_split["train"][:N_PER_SPLIT]
print(f"Using {len(rows_by_split['train'])} training examples")
MODELS, revisions = meta["models"], meta["revisions"]
MAX_TOKENS = meta["max_tokens"]
HIDDEN_SIZES = {"llama": 3072, "qwen": 2048}


Using 5 training examples


In [21]:
import os

from huggingface_hub import hf_hub_download, login, whoami
from huggingface_hub.utils import GatedRepoError

token = os.environ.get("HF_TOKEN")
if token is None:
    from google.colab import userdata
    try:
        token = userdata.get("HF_TOKEN")
    except userdata.SecretNotFoundError as exc:
        raise RuntimeError("Secret 'HF_TOKEN' not found. Add it in the Colab Secrets sidebar.") from exc
    except userdata.NotebookAccessError as exc:
        raise RuntimeError("Notebook access disabled. Toggle on 'Notebook access' for HF_TOKEN.") from exc

login(token=token)
print(f"Authenticated as: {whoami()['name']}")
try:
    hf_hub_download(repo_id=MODELS["llama"], filename="config.json")
except GatedRepoError as exc:
    raise RuntimeError(
        f"Access denied for {MODELS['llama']}. Accept the license on its Hugging Face page."
    ) from exc
print(f"Access confirmed for {MODELS['llama']}")


Authenticated as: siddharthgrover
Access confirmed for meta-llama/Llama-3.2-3B


## Load and label the official validation and test passages
Use the training run’s dataset revision, tokenizers and context cap. The noun filter applies only when sampling training novels.

In [22]:
tokenizers = {
    name: AutoTokenizer.from_pretrained(repo, revision=revisions[name])
    for name, repo in MODELS.items()
}
for tok in tokenizers.values():
    tok.padding_side = "right"
    tok.pad_token = tok.eos_token

In [23]:
WORD = re.compile(r"\S+")
LETTER_WORD = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")


def split_last_word(text):
    """Prefix, continuation, and word count for a passage whose label is its last word."""
    spans = list(WORD.finditer(text))
    if len(spans) < 2 or spans[-1].end() != len(text):
        return None
    if LETTER_WORD.fullmatch(spans[-1].group()) is None:
        return None
    end = spans[-2].end()
    return text[:end], text[end:], len(spans)

In [24]:
def annotate_tokens(row):
    prefix = row["prefix"]
    full = prefix + row["continuation"]
    for name, tok in tokenizers.items():
        ids = tok.encode(prefix)
        complete = tok.encode(full)
        if len(ids) > MAX_TOKENS or complete[:len(ids)] != ids or len(complete) <= len(ids):
            return None
        row[name + "_length"] = len(ids)
        row[name + "_target_id"] = complete[len(ids)]
        row[name + "_target_length"] = len(complete) - len(ids)
    return row


def label_passage(text, example_id, split):
    parts = split_last_word(text)
    if parts is None:
        return None
    prefix, continuation, n_words = parts
    return annotate_tokens(dict(
        id=example_id, split=split, n_words=n_words,
        prefix=prefix, continuation=continuation,
    ))


def passages_to_rows(split, limit):
    rows, dropped = [], 0
    for index, text in enumerate(dataset[split]["text"]):
        row = label_passage(text, f"{split}/{index}", split)
        if row is None:
            dropped += 1
            continue
        rows.append(row)
        if len(rows) == limit:
            break
    print(f"{split}: {len(rows):,} labeled, {dropped:,} dropped")
    return rows


In [25]:
dataset = {s: load_dataset(meta["dataset"], revision=meta["dataset_revision"], split=s)
           for s in ("validation", "test")}
for split in dataset:
    rows_by_split[split] = passages_to_rows(split, N_PER_SPLIT)


validation: 5 labeled, 0 dropped
test: 5 labeled, 0 dropped


## Extract and save last-position vectors

In [26]:
@torch.inference_mode()
def extract(name, model, tokenizer, rows, device):
    collected = {"pre": [], "post": []}
    for start in range(0, len(rows), BATCH_SIZE):
        batch = rows[start:start + BATCH_SIZE]
        inputs = tokenizer([row["prefix"] for row in batch], padding=True,
                           return_tensors="pt").to(device)
        lengths = inputs.attention_mask.sum(1)
        assert lengths.tolist() == [row[name + "_length"] for row in batch]
        idx = (torch.arange(len(batch), device=device), lengths - 1)

        def capture(module, args, output):
            collected["pre"].append(args[0][idx].cpu())
            collected["post"].append(output[idx].cpu())

        hook = model.norm.register_forward_hook(capture)
        try:
            out = model(**inputs, use_cache=False)
        finally:
            hook.remove()
        if start == 0:
            post = collected["post"][-1]
            torch.testing.assert_close(post, out.last_hidden_state[idx].cpu())
            torch.testing.assert_close(model.norm(collected["pre"][-1].to(device)).cpu(), post)
        del out
    return {stage: torch.cat(chunks) for stage, chunks in collected.items()}

In [27]:
def save_tensor_file(payload, path):
    from tempfile import NamedTemporaryFile
    with NamedTemporaryFile(dir=path.parent, delete=False) as tmp:
        temp_path = Path(tmp.name)
    try:
        torch.save(payload, temp_path)
        os.replace(temp_path, path)
    finally:
        temp_path.unlink(missing_ok=True)

In [28]:
def assemble_cache(split, rows, run_dir, fingerprint, provenance):
    ids = [row["id"] for row in rows]
    cache = dict(
        ids=ids,
        target_token_ids=torch.tensor([row["qwen_target_id"] for row in rows]),
        llama_target_token_ids=torch.tensor([row["llama_target_id"] for row in rows]),
        source_model=MODELS["llama"],
        target_model=MODELS["qwen"],
        provenance=provenance,
    )
    for name, width in HIDDEN_SIZES.items():
        part = torch.load(run_dir / f"{name}.pt", map_location="cpu", weights_only=True)
        assert part["ids"] == ids and part["fingerprint"] == fingerprint
        for stage, suffix in [("pre", ""), ("post", "_post")]:
            matrix = part[stage]
            assert matrix.shape == (len(rows), width) and torch.isfinite(matrix).all()
            cache["h_" + name + suffix] = matrix
    cache_path = run_dir / "paired.pt"
    save_tensor_file(cache, cache_path)
    saved = torch.load(cache_path, map_location="cpu", weights_only=True)
    assert saved["ids"] == ids
    for key, value in cache.items():
        if isinstance(value, torch.Tensor):
            assert torch.equal(saved[key], value)
            print(key, tuple(value.shape), value.dtype)
    print("Paired cache:", cache_path)
    return cache_path

In [29]:
rows_by_split = {split: rows[:N_PER_SPLIT] for split, rows in rows_by_split.items()}
for split, rows in rows_by_split.items():
    if len(rows) != N_PER_SPLIT:
        raise ValueError(f"{split} has {len(rows)} examples, expected {N_PER_SPLIT}.")
cache_root = DRIVE_PROJECT_DIR / "activations/lambada" / train_dir.name / f"n{N_PER_SPLIT}"
split_meta = {}
fingerprints = {}
for split, rows in rows_by_split.items():
    directory = cache_root / split
    directory.mkdir(parents=True, exist_ok=True)
    data = "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in rows).encode()
    info = dict(meta, split=split, n_examples=len(rows), manifest_sha256=hashlib.sha256(data).hexdigest())
    if split != "train":
        info = {key: info[key] for key in ("dataset", "dataset_revision", "split", "models", "revisions", "max_tokens", "n_examples", "manifest_sha256")}
    split_meta[split] = info
    fingerprints[split] = hashlib.sha256(json.dumps(info, sort_keys=True).encode()).hexdigest()
    (directory / "examples.jsonl").write_bytes(data)
    print(split, len(rows), "examples ->", directory)

for name, repo in MODELS.items():
    pending = [s for s in rows_by_split if not (cache_root / s / f"{name}.pt").exists()]
    if not pending:
        continue
    if not torch.cuda.is_available():
        raise RuntimeError("Select a CUDA GPU runtime for extraction.")
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    model = AutoModel.from_pretrained(repo, revision=revisions[name], torch_dtype=dtype)
    model = model.eval().requires_grad_(False).to("cuda")
    try:
        for split in pending:
            rows = rows_by_split[split]
            tensors = extract(name, model, tokenizers[name], rows, "cuda")
            assert all(torch.isfinite(v).all() for v in tensors.values())
            save_tensor_file(dict(ids=[r["id"] for r in rows], fingerprint=fingerprints[split], **tensors),
                             cache_root / split / f"{name}.pt")
    finally:
        del model
        gc.collect()
        torch.cuda.empty_cache()

for split, rows in rows_by_split.items():
    assemble_cache(split, rows, cache_root / split, fingerprints[split], split_meta[split])


train 5 examples -> /content/drive/MyDrive/coms-4705-project/activations/lambada/7b6847a7f51353d0/n5/train
validation 5 examples -> /content/drive/MyDrive/coms-4705-project/activations/lambada/7b6847a7f51353d0/n5/validation
test 5 examples -> /content/drive/MyDrive/coms-4705-project/activations/lambada/7b6847a7f51353d0/n5/test


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

target_token_ids (5,) torch.int64
llama_target_token_ids (5,) torch.int64
h_llama (5, 3072) torch.bfloat16
h_llama_post (5, 3072) torch.bfloat16
h_qwen (5, 2048) torch.bfloat16
h_qwen_post (5, 2048) torch.bfloat16
Paired cache: /content/drive/MyDrive/coms-4705-project/activations/lambada/7b6847a7f51353d0/n5/train/paired.pt
target_token_ids (5,) torch.int64
llama_target_token_ids (5,) torch.int64
h_llama (5, 3072) torch.bfloat16
h_llama_post (5, 3072) torch.bfloat16
h_qwen (5, 2048) torch.bfloat16
h_qwen_post (5, 2048) torch.bfloat16
Paired cache: /content/drive/MyDrive/coms-4705-project/activations/lambada/7b6847a7f51353d0/n5/validation/paired.pt
target_token_ids (5,) torch.int64
llama_target_token_ids (5,) torch.int64
h_llama (5, 3072) torch.bfloat16
h_llama_post (5, 3072) torch.bfloat16
h_qwen (5, 2048) torch.bfloat16
h_qwen_post (5, 2048) torch.bfloat16
Paired cache: /content/drive/MyDrive/coms-4705-project/activations/lambada/7b6847a7f51353d0/n5/test/paired.pt
